In [49]:
import numpy as np
import pandas as pd
import json
import argparse

In [67]:
parser = argparse.ArgumentParser(description='Generate TinyStories instruction-tuning datasets and evaluation prompts from csv file')
parser.add_argument('--file_path', type=str, default='policies_responses.csv', help='Path to csv file containing policies, gain-framed responses, and loss-framed responses (default: policies_responses.csv)')
args = parser.parse_args()
file_path = args.file_path

In [53]:
# Import csv with policies and responses
df = pd.read_csv('policies_responses.csv')

# Set seed for reproducibility
rng = np.random.default_rng(100)

# Prompt templates for both instruction and evaluation datasets
prompt_templates = [
    'Should governments invest in {}?',
    'Should policymakers increase funding for {}?',
    'Should public resources be used for {}?',
    'Should governments allocate resources toward {}?',
    'Should the government support {}?',
    'Should the government provide funding for {}?',
    'Should government funds be used for {}?',
    'Should policymakers support {}?'
]

# Separate instruction and evaluation datasets
unique_policies = set(df['policy'])
eval_policies = rng.choice(list(unique_policies), size=40, replace=False).tolist()

train_policies = list(unique_policies - set(eval_policies))
train_set = df[df['policy'].isin(train_policies)]

# Create evaluation prompt dataframe
eval_df = pd.DataFrame(columns=['policy', 'prompt'])

for policy in eval_policies:
    for template in prompt_templates:
        eval_prompt = template.format(policy)

        # Append to eval_df
        new_row = pd.DataFrame([{'policy': policy, 'prompt': eval_prompt}])
        eval_df = pd.concat([eval_df, new_row], ignore_index=True)

eval_df.set_index('policy', inplace=True)

In [55]:
# Create instruction datasets
gain_dataset = []
loss_dataset = []

for _, row in train_set.iterrows():
    policy = row['policy']
    gain_response = row['gain_response']
    loss_response = row['loss_response']

    for template in prompt_templates:
        prompt = template.format(policy)
        
        # Append prompt-response pairs as conversations in the instruction datasets
        gain_dataset.append({
            'conversation': [
                {'name': 'user', 'text': prompt},
                {'name': 'assistant', 'text': gain_response}
            ]
        })

        loss_dataset.append({
            'conversation': [
                {'name': 'user', 'text': prompt},
                {'name': 'assistant', 'text': loss_response}
            ]
        })

# Save datasets
with open('gain_instructions.json', 'w') as f:
    json.dump(gain_dataset, f, indent=2)
print(f'Successfully saved gain-framed instruction set ({len(gain_dataset)} samples).')

with open('loss_instructions.json', 'w') as f:
    json.dump(loss_dataset, f, indent=2)
print(f'Successfully saved loss-framed instruction set ({len(loss_dataset)} samples).')

train_set.to_csv('instruction_responses.csv')
print(f'Successfully saved instruction responses ({len(train_set) * 2} responses).')

eval_df.to_csv('evaluation_prompts.csv')
print(f'Successfully saved evaluation set ({len(eval_df)} prompts).')

Successfully saved gain-framed instruction set (4800 samples).
Successfully saved loss-framed instruction set (4800 samples).
Successfully saved instruction responses (1200 responses).
Successfully saved evaluation set (320 prompts).
